# Notebook 06 — Governance Agent
Applies OWASP LLM Top 10 2025 and NIST AI RMF grounded rules to produce ALLOW/BLOCK/SANITIZE decisions.

In [ ]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    BASE = str(Path(".").resolve().parent)

for p in [os.path.join(BASE, "AGENTS"), os.path.join(Path(".").resolve().parent, "AGENTS")]:
    if p not in sys.path: sys.path.insert(0, p)

CACHE_DIR   = os.path.join(BASE, "EXPERIMENT_CACHE")
MODELS_DIR  = os.path.join(BASE, "MODELS")
RESULTS_DIR = os.path.join(BASE, "RESULTS"); os.makedirs(RESULTS_DIR, exist_ok=True)
RULES_PATH  = os.path.join(BASE, "RULES", "governance_rules.json")
if not os.path.exists(RULES_PATH):
    # try repo root
    RULES_PATH = str(Path(".").resolve().parent / "RULES" / "governance_rules.json")
print(f"Rules path: {RULES_PATH}")
print(f"Rules exist: {os.path.exists(RULES_PATH)}")

In [ ]:
from governance_agent import GovernanceAgent

gov = GovernanceAgent(rules_path=RULES_PATH)
rules = gov.list_rules()
rules_df = pd.DataFrame(rules)
print(f"Loaded {len(rules)} governance rules:")
display(rules_df)

In [ ]:
from risk_agent import RiskAgent

# Load features
features_path = os.path.join(CACHE_DIR, "features.parquet")
assert os.path.exists(features_path), "Run Notebook 04 first!"
df = pd.read_parquet(features_path)
test_df = df[df["split"] == "test"].copy()

# Load risk scores (from risk model if available, else use malicious_probability)
try:
    risk_agent = RiskAgent(models_dir=MODELS_DIR)
    feat_cols = list(pd.read_json(os.path.join(MODELS_DIR, "feature_columns.json"))["columns"])
    features_list = test_df[feat_cols].to_dict("records")
    risk_results = risk_agent.score_batch(features_list, test_df["sample_id"].tolist())
    test_df["risk_score"] = [r.risk_score for r in risk_results]
    print("Risk scores from trained model.")
except Exception as e:
    print(f"Using malicious_probability as risk_score fallback: {e}")
    test_df["risk_score"] = test_df["malicious_probability"]

# Apply governance rules
decisions = []
for _, row in test_df.iterrows():
    d = gov.evaluate(
        risk_score=row["risk_score"],
        vision_score=row.get("vision_score", 0.0),
        hidden_text_detected=row.get("hidden_text_score", 0.0) >= 0.5,
        keyword_density=row.get("keyword_density", 0.0),
        footer_density=row.get("footer_text_density", 0.0),
        watermark_score=row.get("watermark_score", 0.0),
        prompt_score=row["malicious_probability"],
        severity=row.get("severity", "medium") if pd.notna(row.get("severity")) else "medium",
    )
    decisions.append({"sample_id": row["sample_id"], "decision": d.decision,
                      "rule_id": d.rule_id, "severity": d.severity})

dec_df = pd.DataFrame(decisions)
test_df = test_df.merge(dec_df, on="sample_id", how="left")
print(f"Decision distribution:\n{test_df['decision'].value_counts()}")

In [ ]:
# Plot decision distribution by label
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label_val, label_name in zip(axes, [0, 1], ["benign", "malicious"]):
    subset = test_df[test_df["label_int"] == label_val]
    counts = subset["decision"].value_counts()
    colors = {"ALLOW": "#2ecc71", "SANITIZE": "#f39c12", "BLOCK": "#e74c3c"}
    c = [colors.get(k, "gray") for k in counts.index]
    ax.bar(counts.index, counts.values, color=c, edgecolor="white", linewidth=1.5)
    ax.set_title(f"{label_name.upper()} — Decision Distribution", fontweight="bold")
    ax.set_ylabel("Count")
    for bar, v in zip(ax.patches, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, v + 5, f"{v:,}",
                ha="center", va="bottom", fontsize=10)

plt.suptitle("Governance Agent Decision Distribution by Ground Truth",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "06_governance_decisions.png"), bbox_inches="tight", dpi=150)
plt.show()

# Save decisions
test_df.to_csv(os.path.join(RESULTS_DIR, "metrics", "06_governance_decisions.csv"), index=False)
print("Results saved.")

# FNR check: malicious samples that were ALLOWed (worst outcome)
mal = test_df[test_df["label_int"] == 1]
fnr_gov = (mal["decision"] == "ALLOW").mean()
print(f"\nGovernance FNR (malicious \u2192 ALLOW): {fnr_gov:.3%}")
print("This is the most critical safety metric \u2014 lower is better.")